In [ ]:
import os
from PIL import Image
from tqdm import tqdm

def resize_and_save_all(src_folder, dst_folder, size=(256, 256), limit=None):
    """
    Resize and save images from src_folder to dst_folder.
    limit: None processes all images, otherwise processes only first 'limit' images.
    """
    os.makedirs(dst_folder, exist_ok=True)

    count = 0
    for filename in tqdm(sorted(os.listdir(src_folder)), desc=f"Processing {os.path.basename(src_folder)}"):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(src_folder, filename)
            try:
                img = Image.open(img_path).resize(size)
                img.save(os.path.join(dst_folder, filename))
                count += 1
                if limit is not None and count >= limit:
                    break
            except Exception as e:
                print(f"Error processing {filename}: {e}")

# Paths
roi_src = r'G:\Glaucoma\unet\ORIGA\ORIGA\data\roi_images'
roi_dst = r'G:\Glaucoma\unet\ORIGA\ORIGA\data\segmentation_data\images'

cup_src = r'G:\Glaucoma\unet\ORIGA\ORIGA\data\masks\cup'
cup_dst = r'G:\Glaucoma\unet\ORIGA\ORIGA\data\segmentation_data\masks\cup'

disc_src = r'G:\Glaucoma\unet\ORIGA\ORIGA\data\masks\disc'
disc_dst = r'G:\Glaucoma\unet\ORIGA\ORIGA\data\segmentation_data\masks\disc'

# Run for ALL images (set limit=None)
resize_and_save_all(roi_src, roi_dst, size=(256, 256), limit=None)
resize_and_save_all(cup_src, cup_dst, size=(256, 256), limit=None)
resize_and_save_all(disc_src, disc_dst, size=(256, 256), limit=None)



In [ ]:
import torch
import torch.nn as nn

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, dropout_p=0.3):
        super(UNet, self).__init__()

        def CBR(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True)
            )

        self.enc1 = CBR(in_channels, 64)
        self.enc2 = CBR(64, 128)
        self.enc3 = CBR(128, 256)
        self.enc4 = CBR(256, 512)

        self.pool = nn.MaxPool2d(2)

        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = nn.Sequential(
            CBR(512, 256),
            nn.Dropout(p=dropout_p)
        )

        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(
            CBR(256, 128),
            nn.Dropout(p=dropout_p)
        )

        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(
            CBR(128, 64),
            nn.Dropout(p=dropout_p)
        )

        self.final = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        d3 = self.upconv3(e4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.upconv2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.upconv1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.final(d1)


In [ ]:
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch
import os

# 🔧 Custom Dataset Class
class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform

        self.image_filenames = sorted([f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.png'))])
        self.mask_filenames = sorted([f for f in os.listdir(mask_dir) if f.lower().endswith(('.jpg', '.png'))])

        if len(self.image_filenames) != len(self.mask_filenames):
            raise ValueError(f"Images ({len(self.image_filenames)}) and masks ({len(self.mask_filenames)}) count mismatch.")

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_filenames[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_filenames[idx])

        img = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        if self.transform:
            img = self.transform(img)
            mask = self.transform(mask)

        # Normalize mask to [0, 1]
        if mask.max() > 1:
            mask = mask / 255.0

        return img, mask

# 🔄 Transform
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts to [0,1] tensor
])

# 📂 Paths
image_dir = r'G:\Glaucoma\unet\ORIGA\ORIGA\data\segmentation_data\images'
mask_dir = r'G:\Glaucoma\unet\ORIGA\ORIGA\data\segmentation_data\masks\cup'

# 📦 Create Dataset
dataset = SegmentationDataset(image_dir, mask_dir, transform=transform)

# ⚡ Create DataLoader (adjust batch_size for your GPU)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0)  # num_workers>0 for faster loading

# 🔍 Example: get one batch
images, masks = next(iter(dataloader))
print("Image batch shape:", images.shape)  # [B, 3, 256, 256]
print("Mask batch shape:", masks.shape)    # [B, 1, 256, 256]


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import os
from torch.utils.data import DataLoader
from tqdm import tqdm

# ✅ Use the UNet with BN + Dropout
model = UNet().to(device)

# ⚙️ Loss Functions
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs)
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        intersection = (inputs * targets).sum()
        return 1 - (2. * intersection + self.smooth) / (inputs.sum() + targets.sum() + self.smooth)

bce_loss = nn.BCEWithLogitsLoss()
dice_loss = DiceLoss()

def combined_loss(pred, target):
    return bce_loss(pred, target) + dice_loss(pred, target)

# ⚡ Optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 📂 Checkpoint setup
checkpoint_dir = "checkpoints_cup"
os.makedirs(checkpoint_dir, exist_ok=True)

# 🔄 Resume if checkpoint exists
latest_epoch = 0
latest_ckpt_path = None
for file in os.listdir(checkpoint_dir):
    if file.startswith("unet_cup_checkpoint_epoch_") and file.endswith(".pth"):
        epoch_num = int(file.split("_")[-1].split(".")[0])
        if epoch_num > latest_epoch:
            latest_epoch = epoch_num
            latest_ckpt_path = os.path.join(checkpoint_dir, file)

if latest_ckpt_path:
    checkpoint = torch.load(latest_ckpt_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    print(f"✅ Resuming from epoch {start_epoch} (Previous loss: {checkpoint['loss']:.4f})")
else:
    start_epoch = 1
    print("🚀 Starting training from scratch")

# 📦 DataLoader (from the dataset we created earlier)
batch_size = 8
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

# 🔁 Training loop
num_epochs = 100
for epoch in range(start_epoch, start_epoch + num_epochs):
    model.train()
    epoch_loss = 0.0
    
    for images, masks in tqdm(dataloader, desc=f"Epoch {epoch}"):
        images, masks = images.to(device), masks.to(device)

        outputs = model(images)
        loss = combined_loss(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(dataloader)
    print(f"🔁 Epoch {epoch}, Avg Loss: {avg_loss:.4f}")

    # 💾 Save checkpoint
    if epoch % 50 == 0 or epoch == start_epoch + num_epochs - 1:
        ckpt_path = os.path.join(checkpoint_dir, f"unet_cup_checkpoint_epoch_{epoch}.pth")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss,
        }, ckpt_path)
        print(f"💾 Checkpoint saved at: {ckpt_path}")

    # 👀 Visualize predictions
    if epoch % 20 == 0 or epoch == start_epoch + num_epochs - 1:
        model.eval()
        with torch.no_grad():
            sample_images, sample_masks = next(iter(dataloader))
            sample_images, sample_masks = sample_images.to(device), sample_masks.to(device)
            preds = torch.sigmoid(model(sample_images))

        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1)
        plt.imshow(sample_images[0].permute(1, 2, 0).cpu())
        plt.title("Input Image")

        plt.subplot(1, 3, 2)
        plt.imshow(sample_masks[0][0].cpu(), cmap='gray')
        plt.title("Ground Truth")

        plt.subplot(1, 3, 3)
        plt.imshow(preds[0][0].cpu(), cmap='gray')
        plt.title("Predicted Mask")
        plt.show()
        model.train()


In [ ]:
#DISCSSSSSSSS

In [ ]:
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import torch
import os

# 🔧 Folder paths for disc segmentation
image_dir = r'G:\Glaucoma\unet\ORIGA\ORIGA\data\segmentation_data\images'
mask_dir = r'G:\Glaucoma\unet\ORIGA\ORIGA\data\segmentation_data\masks\disc'  # Disc masks path

# ✅ Load all sorted image filenames
image_filenames = sorted([f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.png'))])
mask_filenames = sorted([f for f in os.listdir(mask_dir) if f.lower().endswith(('.jpg', '.png'))])

# Safety check: ensure same count
assert len(image_filenames) == len(mask_filenames), "Mismatch between images and masks count!"

# 🔄 Transform
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts to [0, 1] and [C, H, W]
])

# 📦 Load and transform images and masks
images = []
masks = []

for img_file, mask_file in zip(image_filenames, mask_filenames):
    img_path = os.path.join(image_dir, img_file)
    mask_path = os.path.join(mask_dir, mask_file)

    img = Image.open(img_path).convert("RGB")
    mask = Image.open(mask_path).convert("L")  # Grayscale mask

    img_tensor = transform(img)
    mask_tensor = transform(mask)

    # Normalize mask to [0, 1]
    if mask_tensor.max() > 1:
        mask_tensor = mask_tensor / 255.0

    images.append(img_tensor)
    masks.append(mask_tensor)

# 📚 Stack into tensors of shape [B, C, H, W]
image_batch = torch.stack(images)
mask_batch = torch.stack(masks)

print("Image batch shape:", image_batch.shape)
print("Mask batch shape:", mask_batch.shape)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Define model
model = UNet().to(device)

# 3. Define optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# =========================
# 4. Combined Loss: BCEWithLogits + DiceLoss
# =========================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs)
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        intersection = (inputs * targets).sum()
        return 1 - (2. * intersection + self.smooth) / (inputs.sum() + targets.sum() + self.smooth)

bce_loss = nn.BCEWithLogitsLoss()
dice_loss = DiceLoss()

def combined_loss(pred, target):
    return bce_loss(pred, target) + dice_loss(pred, target)


# =========================
# 5. Resume from latest checkpoint (DISC)
# =========================
checkpoint_dir = "checkpoints_disc"
os.makedirs(checkpoint_dir, exist_ok=True)

latest_epoch = 0
latest_ckpt_path = None

for file in os.listdir(checkpoint_dir):
    if file.startswith("unet_disc_checkpoint_epoch_") and file.endswith(".pth"):
        epoch_num = int(file.split("_")[-1].split(".")[0])
        if epoch_num > latest_epoch:
            latest_epoch = epoch_num
            latest_ckpt_path = os.path.join(checkpoint_dir, file)

if latest_ckpt_path:
    checkpoint = torch.load(latest_ckpt_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    print(f"✅ Resuming from epoch {start_epoch} (Previous loss: {checkpoint['loss']:.4f})")
else:
    start_epoch = 1
    print("🚀 Starting training from scratch")


# =========================
# 6. Load image and mask batch
# =========================
image = image_batch.to(device)
mask = mask_batch.to(device)

# =========================
# 7. Training Loop
# =========================
end_epoch = start_epoch + 100

for epoch in range(start_epoch, end_epoch + 1):
    model.train()
    output = model(image)
    loss = combined_loss(output, mask)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"🔁 Epoch {epoch}, Loss: {loss.item():.4f}")

    # Save full checkpoint
    if epoch % 50 == 0 or epoch == end_epoch:
        ckpt_path = os.path.join(checkpoint_dir, f"unet_disc_checkpoint_epoch_{epoch}.pth")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss.item(),
        }, ckpt_path)
        print(f"💾 Checkpoint saved at: {ckpt_path}")

    # Visualize predictions
    if epoch % 20 == 0 or epoch == end_epoch:
        with torch.no_grad():
            model.eval()
            prob = torch.sigmoid(output)
            print(f"Pred Range (epoch {epoch}):", prob.min().item(), prob.max().item())

            plt.figure(figsize=(12, 4))
            plt.subplot(1, 3, 1)
            plt.imshow(image[0].permute(1, 2, 0).cpu())
            plt.title("Input Image")

            plt.subplot(1, 3, 2)
            plt.imshow(mask[0][0].cpu(), cmap='gray')
            plt.title("Ground Truth")

            plt.subplot(1, 3, 3)
            plt.imshow(prob[0][0].cpu(), cmap='gray')
            plt.title("Predicted Mask")
            plt.show()

        model.train()